# Praat Scripts (日本語版) - 無音・ポーズ区間の自動検出 (Google Colab版) 🎙️

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okawawaka/praat-scripts-ja/blob/main/notebooks/mark_pauses.ipynb)

本ノートブックは、`scripts/segmentation/mark_pauses.praat` と**全く同一のアルゴリズム（To TextGrid (silences)）**で、音声ファイルから無音・ポーズ区間を自動検出し、TextGrid ファイルを自動生成するツールです。

生成された `.TextGrid` ファイルは、**自動的にZIPアーカイブにまとめられて一括ダウンロード**されます。

### Step 1: 必要なライブラリのインストール

In [ ]:
!pip install -q praat-parselmouth

### Step 2: ポーズ検出パラメータの設定

In [ ]:
# @title 【パラメータ設定】 { run: "auto" }
min_pause_duration = 0.2 # @param {type:"number"}
min_sounding_duration = 0.1 # @param {type:"number"}
silence_threshold = -25.0 # @param {type:"number"}
pause_label = "_" # @param {type:"string"}
sounding_label = "sound" # @param {type:"string"}

print(f"設定完了: 最小ポーズ={min_pause_duration}s, 最小発話={min_sounding_duration}s, しきい値={silence_threshold}dB")

### Step 3: 音声ファイル(.wav)をアップロード
下のセルを実行して表示される「ファイル選択」ボタンから、ポーズを検出したい `.wav` ファイル（複数選択可）を選んでください。

In [ ]:
import os
import zipfile
import tempfile
import parselmouth
from parselmouth.praat import call
from google.colab import files

print("▼ 下の『ファイル選択』ボタンをクリックして、.wav ファイルを選択してください：")
uploaded = files.upload()

if not uploaded:
    print("ファイルが選択されませんでした。")
else:
    temp_dir = tempfile.mkdtemp()
    output_dir = os.path.join(temp_dir, "textgrids")
    os.makedirs(output_dir, exist_ok=True)

    for fname, data in uploaded.items():
        if fname.lower().endswith(".wav"):
            with open(os.path.join(temp_dir, fname), "wb") as f:
                f.write(data)

    wav_files = [f for f in uploaded.keys() if f.lower().endswith(".wav")]
    total_processed = 0

    for w_name in wav_files:
        base = os.path.splitext(w_name)[0]
        wav_path = os.path.join(temp_dir, w_name)

        try:
            sound = parselmouth.Sound(wav_path)
            # Praat公式の To TextGrid (silences) を直接実行
            tg = call(sound, "To TextGrid (silences)", silence_threshold, min_pause_duration, min_sounding_duration, pause_label, sounding_label)
            
            out_tg_path = os.path.join(output_dir, base + ".TextGrid")
            call(tg, "Save as text file", out_tg_path)
            total_processed += 1
            print(f"{base}: ポーズ検出 TextGrid を生成しました。")
        except Exception as e:
            print(f"エラー ({base}): {e}")

    print(f"\n処理完了: 合計 {total_processed} 個の TextGrid を生成しました。")

### Step 4: 生成されたTextGridをZIPで一括ダウンロード

In [ ]:
if 'total_processed' in locals() and total_processed > 0:
    zip_path = "detected_textgrids.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files_in_dir in os.walk(output_dir):
            for f in files_in_dir:
                zipf.write(os.path.join(root, f), arcname=f)

    print(f"{zip_path} を作成しました。ダウンロードを開始します...")
    files.download(zip_path)
else:
    print("ダウンロード対象のファイルがありません。")